# 00 - Speaker Diarization

Identifies which segments belong to which speaker (male vs female) using **pyannote.audio**.

**Requirements:**
- Google Colab with GPU
- Free HuggingFace token (get one at https://huggingface.co/settings/tokens)
- Accept model conditions at https://huggingface.co/pyannote/speaker-diarization-community-1

**Output:** Updated translation JSONs with a `speaker` field per segment.

In [ ]:
!pip install -q pyannote.audio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/AIVideoLanguageTransformation'
AUDIO_DIR = os.path.join(PROJECT_DIR, 'data', 'audio', 'original')
TRANSLATIONS_DIR = os.path.join(PROJECT_DIR, 'data', 'translations')

print(f'Audio files: {sorted(os.listdir(AUDIO_DIR))}')
print(f'Translation files: {sorted(os.listdir(TRANSLATIONS_DIR))}')

In [ ]:
# Set your HuggingFace token
# Option 1: Colab Secrets (recommended) — add HF_TOKEN in the key icon on the left sidebar
# Option 2: Paste directly below
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('Loaded HF_TOKEN from Colab Secrets')
except:
    HF_TOKEN = ''  # Paste your token here if not using Colab Secrets
    print('Using hardcoded HF_TOKEN')

In [ ]:
import torch
from pyannote.audio import Pipeline

print('Loading pyannote diarization pipeline...')
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=HF_TOKEN
)
pipeline.to(torch.device('cuda'))
print('Pipeline loaded.')

In [ ]:
def diarize_audio(audio_path, num_speakers=2):
    """Run speaker diarization and return list of (start, end, speaker) tuples."""
    print(f'Diarizing: {os.path.basename(audio_path)}')
    output = pipeline(audio_path, num_speakers=num_speakers)

    speaker_segments = []
    for turn, _, speaker in output.itertracks(yield_label=True):
        speaker_segments.append({
            'start': turn.start,
            'end': turn.end,
            'speaker': speaker
        })
        print(f'  [{turn.start:7.2f}s - {turn.end:7.2f}s] {speaker}')

    return speaker_segments


def assign_speakers(segments, speaker_segments):
    """Assign a speaker label to each transcript segment based on max overlap."""
    for seg in segments:
        best_speaker = 'UNKNOWN'
        best_overlap = 0.0
        for sp in speaker_segments:
            overlap_start = max(seg['start'], sp['start'])
            overlap_end = min(seg['end'], sp['end'])
            overlap = max(0.0, overlap_end - overlap_start)
            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = sp['speaker']
        seg['speaker'] = best_speaker
    return segments

In [ ]:
import json

audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')])

for audio_file in audio_files:
    audio_path = os.path.join(AUDIO_DIR, audio_file)
    stem = os.path.splitext(audio_file)[0]

    # Find matching translation file
    trans_path = os.path.join(TRANSLATIONS_DIR, f'{stem}_en.json')
    if not os.path.exists(trans_path):
        print(f'No translation file for {stem}, skipping')
        continue

    print(f'\n{"="*60}')
    print(f'Processing: {audio_file}')

    # Diarize
    speaker_segments = diarize_audio(audio_path, num_speakers=2)

    # Load translations and assign speakers
    with open(trans_path, 'r', encoding='utf-8') as f:
        segments = json.load(f)

    segments = assign_speakers(segments, speaker_segments)

    # Save updated translations
    with open(trans_path, 'w', encoding='utf-8') as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)

    # Print summary
    speakers = {}
    for seg in segments:
        sp = seg.get('speaker', 'UNKNOWN')
        speakers[sp] = speakers.get(sp, 0) + 1
    print(f'\nSpeaker distribution: {speakers}')
    print(f'Saved: {trans_path}')

print('\nDiarization complete! Now extract voice references for each speaker.')

In [ ]:
# Preview: show first 10 segments with speaker labels for each video
for trans_file in sorted(os.listdir(TRANSLATIONS_DIR)):
    if not trans_file.endswith('_en.json'):
        continue
    with open(os.path.join(TRANSLATIONS_DIR, trans_file), 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'\n=== {trans_file} ===')
    for seg in data[:10]:
        sp = seg.get('speaker', '?')
        print(f"  [{sp}] [{seg['start']:.1f}s-{seg['end']:.1f}s] {seg.get('text_en', seg.get('text_en_deepl', ''))}")

## Extract Voice References Per Speaker

After diarization, extract clean voice samples for each speaker.
Pick segments where only one speaker is talking (no overlap) and the audio is clean.

In [ ]:
from pydub import AudioSegment

VOICE_REF_DIR = os.path.join(PROJECT_DIR, 'data', 'audio', 'voice_reference')
os.makedirs(VOICE_REF_DIR, exist_ok=True)

# Use the first video to extract voice references
# Pick the audio file with the most segments for best reference quality
audio_file = audio_files[0]
audio_path = os.path.join(AUDIO_DIR, audio_file)
stem = os.path.splitext(audio_file)[0]
trans_path = os.path.join(TRANSLATIONS_DIR, f'{stem}_en.json')

with open(trans_path, 'r', encoding='utf-8') as f:
    segments = json.load(f)

audio = AudioSegment.from_wav(audio_path)

# Group segments by speaker, pick longest ones for reference
from collections import defaultdict
speaker_segs = defaultdict(list)
for seg in segments:
    sp = seg.get('speaker', 'UNKNOWN')
    duration = seg['end'] - seg['start']
    speaker_segs[sp].append((duration, seg))

for speaker, segs in speaker_segs.items():
    # Sort by duration, take longest segments up to ~30s total
    segs.sort(key=lambda x: -x[0])
    combined = AudioSegment.empty()
    total = 0
    for dur, seg in segs:
        if total >= 30:
            break
        start_ms = int(seg['start'] * 1000)
        end_ms = int(seg['end'] * 1000)
        combined += audio[start_ms:end_ms]
        total += dur

    ref_path = os.path.join(VOICE_REF_DIR, f'{speaker}_ref.wav')
    combined.export(ref_path, format='wav')
    print(f'Saved voice reference: {ref_path} ({total:.1f}s from {len(segs)} segments)')

print(f'\nVoice references saved to {VOICE_REF_DIR}')
print('Listen to each reference to verify which is male and which is female.')
print('Rename them if needed (e.g., SPEAKER_00_ref.wav → male_ref.wav).')